> **Long run.** Same notebook with numbers of replicas, toy experiments and chain lengths close to those of the report.

# 03 · Frequentist analysis (3/4): systematic uncertainty and stability

**What this notebook does.** The result depends on choices that the data do not fix. They are varied one at a time in
13 variants (window, background model, signal model, preselection), the shift of $A_{\rm raw}$ is compared with the
statistical fluctuation expected between nested samples ($\sigma_B$, Barlow criterion [EXTRA]) and the groups are added
in quadrature. Then the stability against the fit window, the signal region and the PID cuts.

| § | Content | Report section |
|---|---|---|
| 7 | Thirteen variants, combination rule, validation of $\sigma_B$ | Result and systematic uncertainties |
| 8 | Fine scan of the PID cuts, window grid, signal region, PID grid | Systematics (add.: stability scans) |

**Scale.** A guided tour of the method, not the production run: replicas, toy experiments and chains are fewer than in
the report, so numbers can differ slightly from it; next to every result the notebook prints the value of the report
(**report:**). The notebook is self-contained and saves its figures in `figures/` and its numbers in `results/`.
Parts beyond the techniques of the course are tagged **[EXTRA]**.

## 0 · Common setup (data and nominal fit, as in notebook 01)

## 0 · Setup

In [ ]:
import os, time, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
from scipy.stats import norm, chi2
from iminuit import Minuit, cost

import plot_style
plot_style.use()
C = plot_style.COLORS
T0 = time.time()

# Data: put the three .root files of CERN Open Data record 4900 in ./data (or edit the path)
DATA_DIR = "data" if os.path.isdir("data") else "../../opendata-project/Data"
np.random.seed(1)

# figures and numerical results are saved next to the notebook
os.makedirs("figures", exist_ok=True); os.makedirs("results", exist_ok=True)
def save_fig(name):
    """Save the current figure in figures/ and show it."""
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

M_K = 493.677            # charged-kaon mass [MeV]
LO, HI = 5100.0, 5600.0  # nominal fit window of the mass [MeV]

## 1 · Data and preselection

The record contains $B^\pm\to h^\pm h^+h^-$ candidates (2011 data, 7 TeV), one file per magnet polarity. Every
candidate has the momentum, the particle-identification (PID) likelihoods and the charge of the three tracks
$H_1,H_2,H_3$. We read only the branches we need.

In [ ]:
tracks = (1, 2, 3)                     # the three final-state hadrons H1, H2, H3
# branches to read for each track: momentum, PID likelihoods, muon flag, charge
branches = [f"H{i}_{v}" for i in tracks for v in ("PX", "PY", "PZ", "ProbK", "ProbPi", "isMuon", "Charge")]

frames = []
for pol in ("Down", "Up"):             # one file per magnet polarity
    with uproot.open(f"{DATA_DIR}/B2HHH_Magnet{pol}.root") as f:
        d = f["DecayTree"].arrays(branches, library="pd")      # read the tree into a pandas DataFrame
    d["magnet"] = pol.lower()          # remember the polarity of each candidate
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print("candidates in the record:", len(df))

**Preselection.** We want $B^\pm\to K^\pm K^+K^-$: all three tracks must look like kaons and not like pions or muons,

$$\texttt{ProbPi}<0.5\ \wedge\ \texttt{ProbK}>0.5\ \wedge\ \lnot\texttt{isMuon}\quad\text{for each of the three tracks.}$$

The cut flow below gives the effect of each requirement alone and cumulatively (additional result of the report).

In [ ]:
# boolean masks, True when the requirement is satisfied by ALL three tracks
not_muon = np.ones(len(df), bool)
pi_ok = np.ones(len(df), bool)
k_ok = np.ones(len(df), bool)
for i in tracks:
    not_muon &= ~df[f"H{i}_isMuon"].to_numpy().astype(bool)    # not identified as a muon
    pi_ok    &= df[f"H{i}_ProbPi"].to_numpy() < 0.5            # not pion-like
    k_ok     &= df[f"H{i}_ProbK"].to_numpy() > 0.5             # kaon-like

# cut flow: how many candidates pass each requirement alone, and applying them one after the other
print(f"{'requirement':28s} {'alone':>10s} {'cumulative':>11s}")
cum = np.ones(len(df), bool)
cutflow_rows = []
for name, mask in (("not muon (3 tracks)", not_muon), ("ProbPi < 0.5 (3 tracks)", pi_ok), ("ProbK > 0.5 (3 tracks)", k_ok)):
    cum &= mask
    print(f"{name:28s} {mask.sum():10d} {cum.sum():11d}")
    cutflow_rows.append((name, int(mask.sum()), int(cum.sum())))
print("report: 6311517 / 975724 / 28812 alone, 23895 cumulative")
pd.DataFrame(cutflow_rows, columns=["requirement", "alone", "cumulative"]).to_csv("results/01_cutflow.csv", index=False)

**Invariant mass.** We give the kaon mass to the three tracks, sum the four-momenta and take $m^2=E^2-|\vec p|^2$.
The two opposite-sign pairs $m(H_1H_2)$ and $m(H_1H_3)$ are kept for the Dalitz plane ($H_1$ is always the kaon with
charge opposite to the $B$).

We keep a slightly looser sample (`ProbK > 0.3`, `ProbPi < 0.7`) and store the *weakest* PID value of each
candidate: later the systematic study and the PID scan simply re-apply tighter cuts on these columns.

In [ ]:
loose = np.ones(len(df), bool)
for i in tracks:
    loose &= (df[f"H{i}_ProbK"] > 0.3) & (df[f"H{i}_ProbPi"] < 0.7) & ~df[f"H{i}_isMuon"].astype(bool)
d = df[loose].reset_index(drop=True)
del df

def invariant_mass(d, particles):
    """Invariant mass [MeV] of the listed tracks, kaon mass hypothesis."""
    E = sum(np.sqrt(d[f"H{i}_PX"]**2 + d[f"H{i}_PY"]**2 + d[f"H{i}_PZ"]**2 + M_K**2) for i in particles)
    px, py, pz = (sum(d[f"H{i}_{c}"] for i in particles) for c in ("PX", "PY", "PZ"))
    return np.sqrt(np.clip(E**2 - px**2 - py**2 - pz**2, 0, None))

d["m"]  = invariant_mass(d, (1, 2, 3))
d["m12"], d["m13"] = invariant_mass(d, (1, 2)), invariant_mass(d, (1, 3))
d["q"]  = d.H1_Charge + d.H2_Charge + d.H3_Charge              # charge of the B
d["pT"] = np.sqrt(sum(d[f"H{i}_PX"] for i in tracks)**2 + sum(d[f"H{i}_PY"] for i in tracks)**2)
d["p"]  = np.sqrt(d.pT**2 + sum(d[f"H{i}_PZ"] for i in tracks)**2)
d["min_probk"]  = d[[f"H{i}_ProbK" for i in tracks]].min(axis=1)
d["max_probpi"] = d[[f"H{i}_ProbPi" for i in tracks]].max(axis=1)
d = d[np.abs(d.q) == 1]
assert (d.H1_Charge == -d.q).all()          # H1 has the opposite charge of the B

nominal = d[(d.min_probk > 0.5) & (d.max_probpi < 0.5)]
sample = nominal[(nominal.m > LO) & (nominal.m < HI)].reset_index(drop=True)
print(f"preselected: {len(nominal)} (report 23895);  in the window [{LO:.0f},{HI:.0f}]: {len(sample)} (report 19731)")
m_pos = sample.m[sample.q > 0].to_numpy()       # B+
m_neg = sample.m[sample.q < 0].to_numpy()       # B-
n_pos, n_neg = len(m_pos), len(m_neg)

## 2 · Mass model and likelihood

For each charge the mass spectrum is a mixture of signal and background on the window $[5100,5600]$ MeV,

$$f^\pm(m)=N_s^\pm\,S(m;\mu,\sigma)+N_b^\pm\,B^\pm(m),\qquad
N_s^\pm=\tfrac{N_s}{2}(1\mp A),$$

* **signal** $S$: Gaussian, truncated and normalised to the window; $\mu,\sigma$ are *shared* by $B^+$ and $B^-$;
* **background** $B^\pm$: sum of two exponentials, $f\,E(\lambda_1)+(1-f)\,E(\lambda_2)$, with a shape and a yield of
  its own for each charge (the steep exponential describes the rise of the background towards the lower edge);
* **$A$** is a parameter of the fit: it is the *parameter of interest* (POI); the other 11 are *nuisance parameters*.

The likelihood is *extended* (the yields are parameters, the number of candidates is Poisson):

$$-\ln\mathcal L=(N_s+N_b)-\sum_{\pm}\sum_{i\in\pm}\ln\!\big[N_s^\pm S(m_i)+N_b^\pm B^\pm(m_i)\big]+\text{const.}$$

We write the pdfs ourselves (a few lines each) and let `iminuit` build the extended likelihood with
`cost.ExtendedUnbinnedNLL`: the model is a function `model(m, *parameters)` that returns the expected total number
of events and the density of events at $m$.

In [ ]:
def gauss_pdf(m, mu, sigma, lo=LO, hi=HI):
    """Gaussian normalised on [lo, hi]."""
    return norm.pdf(m, mu, sigma) / (norm.cdf(hi, mu, sigma) - norm.cdf(lo, mu, sigma))

def exp_pdf(m, lam, lo=LO, hi=HI):
    """Decreasing exponential exp(-lam (m - lo)) normalised on [lo, hi]."""
    return lam * np.exp(-lam * (m - lo)) / (1 - np.exp(-lam * (hi - lo)))

def bkg_pdf(m, lam1, lam2, f, lo=LO, hi=HI):
    """Background: two exponentials, weight f on the steep one (lam1 > lam2)."""
    return f * exp_pdf(m, lam1, lo, hi) + (1 - f) * exp_pdf(m, lam2, lo, hi)

# one model per charge; the parameters with the same name (N_s, A, mu, sigma) are shared in the joint fit
def model_pos(m, N_s, A, mu, sigma, N_b_pos, lam1_pos, lam2_pos, f_pos):
    n_sig = N_s * (1 - A) / 2                       # B+ : (1 - A) / 2
    return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu, sigma) + N_b_pos * bkg_pdf(m, lam1_pos, lam2_pos, f_pos)

def model_neg(m, N_s, A, mu, sigma, N_b_neg, lam1_neg, lam2_neg, f_neg):
    n_sig = N_s * (1 + A) / 2                       # B- : (1 + A) / 2
    return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu, sigma) + N_b_neg * bkg_pdf(m, lam1_neg, lam2_neg, f_neg)

### 3.2 Simultaneous fit: the nominal result

The two charges are fitted *together*: the total cost is the sum of the two extended likelihoods and $A$ is a
parameter of it, so the profile likelihood of $A$ is available directly. `Migrad` minimises, `Hesse` gives the
symmetric uncertainties from the curvature (inverse of the Fisher information) and `Minos` the asymmetric
intervals from the profile likelihood.

In [ ]:
def fit_joint(m_p, m_n, start, a_fixed=None):
    """Simultaneous fit of B+ and B-: the cost is the sum of the two extended likelihoods.
    With a_fixed the asymmetry is held at that value (profile likelihood at that point)."""
    nll = cost.ExtendedUnbinnedNLL(m_p, model_pos) + cost.ExtendedUnbinnedNLL(m_n, model_neg)
    f = Minuit(nll, **start)
    f.limits["A"] = (-1, 1); f.limits["mu"] = (5260, 5310); f.limits["sigma"] = (3, 40)
    f.limits["N_s"] = (0, 2 * (len(m_p) + len(m_n)))
    for s in ("pos", "neg"):
        f.limits[f"N_b_{s}"] = (0, 2 * (len(m_p) + len(m_n)))
        f.limits[f"lam1_{s}"] = (1e-5, 0.1); f.limits[f"lam2_{s}"] = (1e-5, 0.1); f.limits[f"f_{s}"] = (0, 1)
    if a_fixed is not None:
        f.values["A"] = a_fixed
        f.fixed["A"] = True
    f.migrad()
    return f

start = dict(N_s=0.78 * len(sample), A=-0.04, mu=5284.0, sigma=18.0,
             N_b_pos=0.22 * n_pos, lam1_pos=0.03, lam2_pos=0.002, f_pos=0.15,
             N_b_neg=0.22 * n_neg, lam1_neg=0.03, lam2_neg=0.002, f_neg=0.15)
fit = fit_joint(m_pos, m_neg, start)
best_vals_start = {k: fit.values[k] for k in start}      # best-fit values, used as starting point of later fits
fit.migrad(); fit.hesse()
fit.minos("A")                                      # 68 % interval (default cl = 1 sigma)
ci68 = (fit.values["A"] + fit.merrors["A"].lower, fit.values["A"] + fit.merrors["A"].upper)
fit_95 = copy.deepcopy(fit); fit_95.minos("A", cl=0.95)
ci95 = (fit_95.values["A"] + fit_95.merrors["A"].lower, fit_95.values["A"] + fit_95.merrors["A"].upper)

assert fit.valid and fit.values["lam1_pos"] > fit.values["lam2_pos"] and fit.values["lam1_neg"] > fit.values["lam2_neg"]
for k in ("N_s", "A", "mu", "sigma"):
    print(f"{k:8s} = {fit.values[k]:12.4f} +- {fit.errors[k]:.4f}")
print(f"N_b = {fit.values['N_b_pos'] + fit.values['N_b_neg']:.0f}")
print(f"\nA_raw = {fit.values['A']:+.4f} +- {fit.errors['A']:.4f} (Hesse)")
print(f"  68% Minos [{ci68[0]:+.4f}, {ci68[1]:+.4f}]    report: [-0.0526, -0.0356]")
print(f"  95% Minos [{ci95[0]:+.4f}, {ci95[1]:+.4f}]    report: [-0.0607, -0.0275]")
print("report: A = -0.0441 +- 0.0085, N_s = 15328 +- 133")

**Reading the result.** $A_{\rm raw}\approx-0.044$ with a Hesse uncertainty of $0.0085$: the asymmetry is about five standard deviations from zero. The Minos intervals are the ranges where the profile likelihood is within $1$ and $1.96^2$ units of its minimum; they are slightly asymmetric and the 95 % one does not contain zero.

In [ ]:
# table of the 12 parameters of the nominal fit, with their Hesse uncertainties, saved for later use
par_table = pd.DataFrame({"value": [fit.values[k] for k in fit.parameters], "hesse": [fit.errors[k] for k in fit.parameters]},
                         index=list(fit.parameters))
par_table.to_csv("results/01_fit_parameters.csv")
print(par_table.round(5).to_string())

### 3.3 Goodness of fit and picture

The $\chi^2=\sum_i(n_i-\nu_i)^2/\nu_i$ in bins of 10 MeV (both charges, $\nu_i$ from the fitted model) with
$\mathrm{ndof}=2\times50-12$. The pulls $(n_i-\nu_i)/\sqrt{\nu_i}$ show *where* the model fails. (The model is not
perfect: the signal tails are heavier than a Gaussian; $A_{\rm raw}$ does not depend on it, see §7.)

In [ ]:
def expected_counts(model, args, edges, n_sub=5):
    """Expected counts per bin: mean of the density at n_sub points in each bin, times the width."""
    width = edges[1] - edges[0]
    offs = (np.arange(n_sub) + 0.5) / n_sub                       # sub-points inside a bin
    pts = edges[:-1, None] + offs[None, :] * width                # shape (n_bins, n_sub)
    return model(pts, *args)[1].mean(axis=1) * width

edges = np.arange(LO, HI + 1e-9, 10.0)                            # bins of 10 MeV
centers = 0.5 * (edges[1:] + edges[:-1])
v = fit.values
# expected number of events in each bin for B+ and B-, from the fitted model
nu_pos = expected_counts(model_pos, [v[k] for k in ("N_s", "A", "mu", "sigma", "N_b_pos", "lam1_pos", "lam2_pos", "f_pos")], edges)
nu_neg = expected_counts(model_neg, [v[k] for k in ("N_s", "A", "mu", "sigma", "N_b_neg", "lam1_neg", "lam2_neg", "f_neg")], edges)
obs_pos, obs_neg = np.histogram(m_pos, edges)[0], np.histogram(m_neg, edges)[0]    # observed counts
chi2_val = np.sum((obs_pos - nu_pos)**2 / nu_pos) + np.sum((obs_neg - nu_neg)**2 / nu_neg)   # Pearson chi2
ndof = 2 * len(centers) - 12                                      # bins of both charges minus parameters
print(f"chi2/ndof = {chi2_val:.1f}/{ndof},  p = {chi2.sf(chi2_val, ndof):.1e}     (report: 160/88, p = 4.8e-6)")

# data with the fit (top) and pulls (bottom): the pulls show WHERE the model fails
pd.DataFrame({"m_center": centers, "obs_pos": obs_pos, "model_pos": nu_pos, "obs_neg": obs_neg, "model_neg": nu_neg}).to_csv(
    "results/01_mass_fit_bins.csv", index=False)    # data and model per bin
fig, ax = plt.subplots(2, 2, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for col, (obs, nu, lab, c) in enumerate(((obs_pos, nu_pos, "$B^+$", C["pos"]), (obs_neg, nu_neg, "$B^-$", C["neg"]))):
    ax[0, col].errorbar(centers, obs, np.sqrt(obs), fmt="o", ms=3, color="k", label="data")
    ax[0, col].plot(centers, nu, color=C["model"], label="fit")
    ax[0, col].set(yscale="log", ylabel="candidates / 10 MeV", title=lab); ax[0, col].legend()
    ax[1, col].bar(centers, (obs - nu) / np.sqrt(nu), width=9, color=c)
    ax[1, col].set(xlabel="m(KKK) [MeV]", ylabel="pull")
plt.tight_layout(); save_fig("03_joint_fit_pulls")

**Reading the result.** $\chi^2/{\rm ndof}$ is well above 1 and the $p$-value tiny: the model is **not** a perfect description of the spectrum (the pulls show structure on the flanks of the peak, where the signal tails are heavier than a Gaussian). What matters for the measurement is whether this affects $A$: the signal-model variants of notebook 03 say that it does not.

In [ ]:
# purity of the signal window |m - mu| < 2 sigma: signal / (signal + background) in the window, from the fit
mu_f, sg_f = fit.values["mu"], fit.values["sigma"]
x_w = np.linspace(mu_f - 2 * sg_f, mu_f + 2 * sg_f, 2001)
sig_in = fit.values["N_s"] * np.trapezoid(gauss_pdf(x_w, mu_f, sg_f), x_w)
bkg_in = sum(fit.values[f"N_b_{s}"] * np.trapezoid(bkg_pdf(x_w, fit.values[f"lam1_{s}"], fit.values[f"lam2_{s}"], fit.values[f"f_{s}"]), x_w)
             for s in ("pos", "neg"))
print(f"signal in the window: {sig_in:.0f}, background: {bkg_in:.0f}, purity {sig_in / (sig_in + bkg_in):.3f}   (report: about 96 %)")

In [ ]:
best_vals = dict(best_vals_start)      # best-fit values of the nominal fit, starting point of the later fits

### 3.4 Counting with sideband subtraction (a model-light cross-check) [EXTRA]

A different, simpler estimate: fit the *background alone* on the sidebands (window without
$[5200,5350]$ MeV), extrapolate it under the peak to get the expected background $b$ in the signal region, and
subtract it from the observed number of candidates. For each charge $S=N_{\rm obs}-b$, then
$A=(S^--S^+)/(S^-+S^+)$.

In [ ]:
SR = (5200.0, 5350.0)                                        # signal region
grid = np.linspace(LO, HI, 5001)

def sideband_background(m_side, sr=SR):
    """Fit of the two-exponential background on the sidebands (window without sr); returns the expected
    background in the signal region sr and the number of sideband events."""
    in_sb = (grid < sr[0]) | (grid > sr[1])
    in_sr = ~in_sb
    m_sb = m_side[(m_side < sr[0]) | (m_side > sr[1])]
    def pdf_sb(m, lam1, lam2, f):                           # background pdf renormalised on the sidebands
        norm_sb = np.trapezoid(bkg_pdf(grid, lam1, lam2, f)[in_sb], grid[in_sb])
        return bkg_pdf(m, lam1, lam2, f) / norm_sb
    f = Minuit(cost.UnbinnedNLL(m_sb, pdf_sb), lam1=0.03, lam2=0.002, f=0.15)
    f.limits["lam1"] = (1e-5, 0.1); f.limits["lam2"] = (1e-5, 0.1); f.limits["f"] = (0, 1)
    f.migrad()
    p = bkg_pdf(grid, *f.values)
    frac_sr = np.trapezoid(p[in_sr], grid[in_sr]) / np.trapezoid(p[in_sb], grid[in_sb])
    return len(m_sb) * frac_sr, len(m_sb)

S, dS = {}, {}
for lab, m_side in (("pos", m_pos), ("neg", m_neg)):
    b, n_sb = sideband_background(m_side)
    n_obs = np.sum((m_side > SR[0]) & (m_side < SR[1]))
    S[lab], dS[lab] = n_obs - b, np.sqrt(n_obs + b**2 / n_sb)
tot = S["pos"] + S["neg"]
a_sb = (S["neg"] - S["pos"]) / tot
da_sb = 2 * np.sqrt((S["pos"] * dS["neg"])**2 + (S["neg"] * dS["pos"])**2) / tot**2
print(f"A_raw (sideband subtraction) = {a_sb:+.4f} +- {da_sb:.4f}     (report: -0.0503 +- 0.0083)")

## 7 · Systematic uncertainty

The nominal result depends on choices that the data do not fix. We vary them **one at a time** (13 variants in four
groups), refit and look at the shift $\Delta=A^{\rm var}-A^{\rm nom}$:

| group | variants |
|---|---|
| window | lower edge 5050, 5150 MeV; upper edge 5450, 5700 MeV |
| background | single exponential; shape shared by the two charges; charge-symmetric background ($A_b=0$) |
| signal | Crystal Ball instead of Gaussian; $\mu,\sigma$ separate for each charge |
| preselection | `ProbK` > 0.4, 0.6; `ProbPi` < 0.4, 0.6 |

Each variant is a small change of the model of §2, so each one gets its own two short model functions (one per charge)
written out explicitly below; the fit of the nominal model, for any window, is `fit_nominal`.

In [ ]:
from iminuit.util import describe              # names of the arguments of a function

def make_models(lo=LO, hi=HI):
    """The nominal models of B+ and B- for the window [lo, hi]: the functions of section 2, for any window."""
    def model_pos(m, N_s, A, mu, sigma, N_b_pos, lam1_pos, lam2_pos, f_pos):
        n_sig = N_s * (1 - A) / 2
        return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu, sigma, lo, hi) + N_b_pos * bkg_pdf(m, lam1_pos, lam2_pos, f_pos, lo, hi)
    def model_neg(m, N_s, A, mu, sigma, N_b_neg, lam1_neg, lam2_neg, f_neg):
        n_sig = N_s * (1 + A) / 2
        return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu, sigma, lo, hi) + N_b_neg * bkg_pdf(m, lam1_neg, lam2_neg, f_neg, lo, hi)
    return model_pos, model_neg

def start_and_limits(name, n):
    """Starting value and limits of a parameter, decided by its name (n = number of candidates)."""
    if name == "N_s":              return 0.78 * n, (0, 2 * n)
    if name == "N_b":              return 0.22 * n, (0, 2 * n)               # background yield shared by the charges
    if name.startswith("N_b_"):    return 0.11 * n, (0, 2 * n)               # background yield of one charge
    if name == "A":                return -0.04, (-1, 1)
    if name.startswith("mu"):      return 5284.0, (5260, 5310)
    if name.startswith("sigma"):   return 18.0, (3, 40)
    if name == "beta":             return 1.8, (0.3, 5)                      # Crystal Ball tail
    if name == "m_cb":             return 10.0, (1.01, 50)
    if name.startswith("lam1"):    return 0.03, (1e-5, 0.1)                  # steep exponential
    if name.startswith("lam2"):    return 0.002, (1e-5, 0.1)                 # slow exponential
    if name.startswith("lam_"):    return 0.004, (1e-5, 0.1)                 # single exponential
    if name.startswith("f"):       return 0.15, (0, 1)
    raise ValueError(name)

def fit_two_charges(model_p, model_n, s):
    """Simultaneous fit of the two charges of the sample s with the given models (parameters with the same name are shared)."""
    mp, mn = s.m[s.q > 0].to_numpy(), s.m[s.q < 0].to_numpy()
    nll = cost.ExtendedUnbinnedNLL(mp, model_p) + cost.ExtendedUnbinnedNLL(mn, model_n)
    names_ = list(dict.fromkeys(describe(model_p)[1:] + describe(model_n)[1:]))        # parameters of the two models
    f = Minuit(nll, **{k: start_and_limits(k, len(s))[0] for k in names_})
    for k in names_:
        f.limits[k] = start_and_limits(k, len(s))[1]
    f.migrad(); f.hesse()
    return f

def fit_nominal(s, lo=LO, hi=HI):
    """Fit of the nominal model to the sample s in the window [lo, hi]."""
    return fit_two_charges(*make_models(lo, hi), s)

# check: this gives the nominal result of section 3
f_nom = fit_nominal(sample)
print(f"fit_nominal: A = {f_nom.values['A']:+.5f}   nominal fit: A = {fit.values['A']:+.5f}")
assert abs(f_nom.values["A"] - fit.values["A"]) < 1e-4

In [ ]:
# ---- variants of the BACKGROUND model: one exponential, same shape for the two charges, charge-symmetric background
def model_pos_1exp(m, N_s, A, mu, sigma, N_b_pos, lam_pos):
    n_sig = N_s * (1 - A) / 2
    return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu, sigma) + N_b_pos * exp_pdf(m, lam_pos)
def model_neg_1exp(m, N_s, A, mu, sigma, N_b_neg, lam_neg):
    n_sig = N_s * (1 + A) / 2
    return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu, sigma) + N_b_neg * exp_pdf(m, lam_neg)

def model_pos_shared(m, N_s, A, mu, sigma, N_b_pos, lam1, lam2, f):         # lam1, lam2, f: no suffix -> shared
    n_sig = N_s * (1 - A) / 2
    return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu, sigma) + N_b_pos * bkg_pdf(m, lam1, lam2, f)
def model_neg_shared(m, N_s, A, mu, sigma, N_b_neg, lam1, lam2, f):
    n_sig = N_s * (1 + A) / 2
    return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu, sigma) + N_b_neg * bkg_pdf(m, lam1, lam2, f)

def model_pos_sym(m, N_s, A, mu, sigma, N_b, lam1_pos, lam2_pos, f_pos):    # N_b shared: N_b/2 for each charge
    n_sig = N_s * (1 - A) / 2
    return n_sig + N_b / 2, n_sig * gauss_pdf(m, mu, sigma) + N_b / 2 * bkg_pdf(m, lam1_pos, lam2_pos, f_pos)
def model_neg_sym(m, N_s, A, mu, sigma, N_b, lam1_neg, lam2_neg, f_neg):
    n_sig = N_s * (1 + A) / 2
    return n_sig + N_b / 2, n_sig * gauss_pdf(m, mu, sigma) + N_b / 2 * bkg_pdf(m, lam1_neg, lam2_neg, f_neg)

# ---- variants of the SIGNAL model: Crystal Ball (Gaussian core with a low-mass tail), mu and sigma per charge
from scipy.stats import crystalball
def cb_pdf(m, mu, sigma, beta, m_cb):
    """Crystal Ball normalised on the window."""
    z = crystalball.cdf((HI - mu) / sigma, beta, m_cb) - crystalball.cdf((LO - mu) / sigma, beta, m_cb)
    return crystalball.pdf((m - mu) / sigma, beta, m_cb) / sigma / z

def model_pos_cb(m, N_s, A, mu, sigma, beta, m_cb, N_b_pos, lam1_pos, lam2_pos, f_pos):
    n_sig = N_s * (1 - A) / 2
    return n_sig + N_b_pos, n_sig * cb_pdf(m, mu, sigma, beta, m_cb) + N_b_pos * bkg_pdf(m, lam1_pos, lam2_pos, f_pos)
def model_neg_cb(m, N_s, A, mu, sigma, beta, m_cb, N_b_neg, lam1_neg, lam2_neg, f_neg):
    n_sig = N_s * (1 + A) / 2
    return n_sig + N_b_neg, n_sig * cb_pdf(m, mu, sigma, beta, m_cb) + N_b_neg * bkg_pdf(m, lam1_neg, lam2_neg, f_neg)

def model_pos_percharge(m, N_s, A, mu_pos, sigma_pos, N_b_pos, lam1_pos, lam2_pos, f_pos):    # own mu, sigma
    n_sig = N_s * (1 - A) / 2
    return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu_pos, sigma_pos) + N_b_pos * bkg_pdf(m, lam1_pos, lam2_pos, f_pos)
def model_neg_percharge(m, N_s, A, mu_neg, sigma_neg, N_b_neg, lam1_neg, lam2_neg, f_neg):
    n_sig = N_s * (1 + A) / 2
    return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu_neg, sigma_neg) + N_b_neg * bkg_pdf(m, lam1_neg, lam2_neg, f_neg)

In [ ]:
def binned_chi2(f, models, s, lo=LO, hi=HI, width=10.0):
    """Pearson chi2 in bins of 10 MeV, both charges, of a fit f made with the pair of models."""
    edges = np.arange(lo, hi + 1e-9, width)
    chi2_ = 0.0
    for model, mm in zip(models, (s.m[s.q > 0], s.m[s.q < 0])):
        nu = expected_counts(model, [f.values[k] for k in describe(model)[1:]], edges)   # expected counts from the fit
        obs = np.histogram(mm, edges)[0]                                                  # observed counts
        chi2_ += np.sum((obs - nu)**2 / nu)
    return chi2_

**Combination rule.**
1. A variant that *changes the sample* (window, preselection) gives a nested sample: part of the shift is a
   statistical fluctuation of expected size $\sigma_B=\sqrt{|\sigma_{\rm var}^2-\sigma_{\rm nom}^2|}$ (Barlow
   criterion) [EXTRA]. It contributes only if $|\Delta|>\sigma_B$.
2. A variant rejected by the data ($\Delta\chi^2>25$ on the same sample, the single exponential) is excluded.
3. In each group the largest remaining $|\Delta|$ is the group uncertainty; groups are independent choices and are
   added in quadrature.

In [ ]:
def select(lo=LO, hi=HI, k=0.5, pi=0.5):
    """Candidates of the loose sample d with mass in (lo, hi), min ProbK > k and max ProbPi < pi."""
    s = d[(d.m > lo) & (d.m < hi) & (d.min_probk > k) & (d.max_probpi < pi)]
    return s.reset_index(drop=True)

# the 13 variants: name -> (group, change of the SAMPLE, pair of models of the B+ and B- samples)
nominal_models = make_models()
VARIANTS = {
    "window_lo5050":       ("window", dict(lo=5050.0), make_models(5050.0, HI)),
    "window_lo5150":       ("window", dict(lo=5150.0), make_models(5150.0, HI)),
    "window_hi5450":       ("window", dict(hi=5450.0), make_models(LO, 5450.0)),
    "window_hi5700":       ("window", dict(hi=5700.0), make_models(LO, 5700.0)),
    "bkg_single_exp":      ("background", {}, (model_pos_1exp, model_neg_1exp)),
    "bkg_shared_shape":    ("background", {}, (model_pos_shared, model_neg_shared)),
    "bkg_symmetric":       ("background", {}, (model_pos_sym, model_neg_sym)),
    "signal_crystal_ball": ("signal", {}, (model_pos_cb, model_neg_cb)),
    "signal_per_charge":   ("signal", {}, (model_pos_percharge, model_neg_percharge)),
    "presel_probk_0.4":    ("preselection", dict(k=0.4), nominal_models),
    "presel_probk_0.6":    ("preselection", dict(k=0.6), nominal_models),
    "presel_probpi_0.4":   ("preselection", dict(pi=0.4), nominal_models),
    "presel_probpi_0.6":   ("preselection", dict(pi=0.6), nominal_models),
}
SAMPLE_CHANGING = ("window", "preselection")     # these variants fit a different (nested) sample: sigma_B applies
# values of A of the report for each variant, printed next to ours
REPORT_A = {"window_lo5050": -0.0445, "window_lo5150": -0.0446, "window_hi5450": -0.0442, "window_hi5700": -0.0455,
            "bkg_single_exp": -0.0481, "bkg_shared_shape": -0.0481, "bkg_symmetric": -0.0472,
            "signal_crystal_ball": -0.0447, "signal_per_charge": -0.0442,
            "presel_probk_0.4": -0.0473, "presel_probk_0.6": -0.0384, "presel_probpi_0.4": -0.0421, "presel_probpi_0.6": -0.0448}

A0, s0 = f_nom.values["A"], f_nom.errors["A"]                  # nominal result and its statistical error
chi2_nom = binned_chi2(f_nom, nominal_models, sample)
syst_rows = []
for name, (group, sel_opt, models) in VARIANTS.items():
    s = select(**sel_opt)                                      # sample of the variant
    f = fit_two_charges(*models, s)                            # fit of the model of the variant
    shift, sig = f.values["A"] - A0, f.errors["A"]             # shift of A with respect to the nominal fit
    changes = group in SAMPLE_CHANGING
    # statistical fluctuation expected between nested samples (Barlow): only for variants that change the sample
    sigma_B = np.sqrt(abs(sig**2 - s0**2)) if changes else np.nan
    # a variant on the same sample that the data reject (chi2 worse by more than 25) is excluded
    rejected = (not changes) and (binned_chi2(f, models, s) - chi2_nom > 25)
    # the shift contributes if the variant is not rejected and, when the sample changes, exceeds sigma_B
    counts = (not rejected) and ((not changes) or abs(shift) > sigma_B)
    syst_rows.append(dict(variant=name, group=group, n=len(s), A=f.values["A"], shift=shift, sigma_B=sigma_B,
                          rejected=rejected, counts=counts, report_A=REPORT_A[name]))
syst = pd.DataFrame(syst_rows)
print(syst.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

# uncertainty of each group = largest contributing shift; the groups are independent: quadrature sum
group_unc = syst[syst.counts].groupby("group")["shift"].apply(lambda x: x.abs().max())
group_unc = group_unc.reindex(["window", "background", "signal", "preselection"]).fillna(0.0)
syst_total = np.sqrt((group_unc**2).sum())
print("\nuncertainty by group:", group_unc.round(4).to_dict())
print(f"total systematic = {syst_total:.4f}   (report: 0.0014, 0.0040, 0.0006, 0.0058 -> 0.0072)")
print(f"\nResult:  A_raw = {A0:+.4f} +- {s0:.4f} (stat) +- {syst_total:.4f} (syst)     report: -0.0441 +- 0.0085 +- 0.0072")

In [ ]:
# shifts of the 13 variants with the statistical spread sigma_B of the sample-changing ones (open marker: rejected)
fig, ax = plt.subplots(figsize=(7, 6))
yy = np.arange(len(syst))[::-1]
for y_, (_, r) in zip(yy, syst.iterrows()):
    ax.errorbar(r["shift"], y_, xerr=None if np.isnan(r.sigma_B) else r.sigma_B, fmt="o", color=C["freq"], capsize=3,
                mfc="none" if r.rejected else C["freq"])
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels(syst.variant)
for sep in (3.5, 6.5, 8.5):                       # boundaries between the four groups (window, background, signal, preselection)
    ax.axhline(len(syst) - 1 - sep, color="0.7", ls=":")
ax.set(xlabel="shift $A^{var}-A^{nom}$ (bar: $\\sigma_B$)")
plt.tight_layout(); save_fig("03_variant_shifts")

**Validation of the $\sigma_B$ rule on nested samples** [EXTRA]. The rule assumes that, between a sample and a sub-sample
of it, the statistical spread of the shift is $\sqrt{|\sigma^2_{\rm sub}-\sigma^2_{\rm full}|}$. We check it with a
**bootstrap**: resample the nominal candidates with replacement, take as sub-sample those with $\texttt{ProbK}>0.6$,
fit both and look at the spread of the shift divided by the median $\sigma_B$ (it should be $\approx1$).
*Report:* 300 replicas, width compatible with 1. Here 20.

In [ ]:
shifts, sigmas = [], []
for r in range(200):
    idx = np.random.choice(len(sample), len(sample), replace=True)     # bootstrap replica of the nominal sample
    full = sample.iloc[idx]
    sub = full[full.min_probk > 0.6]                                   # nested sub-sample: tighter kaon requirement
    f_full = fit_joint(full.m[full.q > 0].to_numpy(), full.m[full.q < 0].to_numpy(), best_vals); f_full.hesse()
    f_sub = fit_joint(sub.m[sub.q > 0].to_numpy(), sub.m[sub.q < 0].to_numpy(), best_vals); f_sub.hesse()
    shifts.append(f_sub.values["A"] - f_full.values["A"])              # shift between the two nested fits
    sigmas.append(np.sqrt(abs(f_sub.errors["A"]**2 - f_full.errors["A"]**2)))     # sigma_B of this replica
shifts, sigmas = np.array(shifts), np.array(sigmas)
print(f"spread of the shift / median sigma_B = {shifts.std() / np.median(sigmas):.2f}   (expected ~ 1;  mean shift {shifts.mean():+.4f})")
print("note: sigma_B is the difference of two nearly equal uncertainties, hence sensitive to small differences between methods:")
print("      a variant at the threshold can be classified differently without changing the result")

**Reading the result.** The spread of the shift divided by $\sigma_B$ is of order 1: the rule describes the fluctuation between nested samples, so a shift larger than $\sigma_B$ is more than a statistical fluctuation of the sub-sample.

In [ ]:
# the shifts between nested samples in units of sigma_B should have unit width
fig, ax = plt.subplots(figsize=(6, 4))
z_sb = shifts / np.median(sigmas)
ax.hist(z_sb, bins=10, density=True, color=C["freq"], alpha=0.6, label=f"{len(z_sb)} bootstrap replicas")
x = np.linspace(-4, 4, 200); ax.plot(x, norm.pdf(x), "k--", label="$N(0,1)$")
ax.set(xlabel="shift / median $\\sigma_B$"); ax.legend()
plt.tight_layout(); save_fig("03_sigmaB_validation")
pd.DataFrame({"shift": shifts, "sigma_B": sigmas}).to_csv("results/03_sigmaB_validation.csv", index=False)

The **preselection** dominates through one single variant, `ProbK > 0.6`. Section 8 asks whether it is a trend
or a fluctuation.

## 8 · Stability scans

### 8.1 Fine scan of the PID cuts, one at a time

Each sample is nested in (or contains) the nominal one, so the shift is compared with $\sigma_B$. A trend with the
cut would indicate a real dependence (e.g. misidentified decays entering when the cut is loosened); a scatter
around zero within $\sim1\sigma_B$ a fluctuation. The report scans in steps of 0.05, as here.

In [ ]:
scan = []
# one cut at a time: ProbK > cut (0.30 ... 0.85) and ProbPi < cut (0.15 ... 0.70), steps of 0.05, nominal skipped
for var, grid_ in (("k", np.arange(0.30, 0.851, 0.05)), ("pi", np.arange(0.15, 0.701, 0.05))):
    for cut in grid_:
        if abs(cut - 0.5) < 1e-9: continue
        s = select(**{var: float(cut)})                        # nested sample (or one that contains the nominal)
        f = fit_nominal(s)
        sig_B = np.sqrt(abs(f.errors["A"]**2 - s0**2))         # expected statistical spread of the shift
        scan.append(dict(var=var, cut=cut, n=len(s), shift=f.values["A"] - A0, sigma_B=sig_B))
scan = pd.DataFrame(scan)

# shift of A against the cut, with sigma_B as error bar: a trend would be a real dependence, scatter a fluctuation
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, (var, lab, nominal_cut) in zip(ax, (("k", "ProbK > cut", 0.5), ("pi", "ProbPi < cut", 0.5))):
    g = scan[scan["var"] == var]
    a.errorbar(g.cut, g["shift"], g.sigma_B, fmt="o", color=C["freq"], capsize=3)
    a.axhline(0, color="k", lw=0.8); a.axvline(nominal_cut, color="gray", ls="-.")
    a.set(xlabel=lab, ylabel="$A^{var}-A^{nom}$")
plt.tight_layout(); save_fig("03_pid_scan")
g = scan[(scan["var"] == "k") & (scan.cut > 0.5)]
print("tight ProbK: shifts / sigma_B =", (g["shift"] / g.sigma_B).round(2).tolist(), "  (report: scatter within 1.3 sigma_B, no trend)")
g = scan[(scan["var"] == "k") & (scan.cut < 0.5)]
print("loose ProbK: shifts           =", g["shift"].round(4).tolist(), "  (report: monotonic)")

**Reading the result.** On the tight side the shifts scatter around zero within about one $\sigma_B$ without a trend: the large shift of `ProbK > 0.6` is a **fluctuation**. Loosening `ProbK` moves $A$ monotonically, as expected if misidentified decays (not in the mass model) enter the sample: this supports the nominal requirement.

### 8.2 Stability against the fit window (grid of the two edges)

In [ ]:
lefts, rights = [5050.0, 5100.0, 5150.0, 5200.0], [5450.0, 5500.0, 5600.0, 5700.0]   # lower and upper edges
grid_A = np.zeros((len(lefts), len(rights)))
for i, lo_ in enumerate(lefts):
    for j, hi_ in enumerate(rights):
        f = fit_nominal(select(lo=lo_, hi=hi_), lo_, hi_)             # sample AND model with the new window
        grid_A[i, j] = f.values["A"]
print("A_raw over the grid of the window edges (rows: lower edge, columns: upper edge)")
print(pd.DataFrame(grid_A, index=lefts, columns=rights).round(4).to_string())
print(f"max |shift| = {np.abs(grid_A - A0).max():.4f}   (report, wider grid: 0.0048)")

### 8.3 Stability against the signal region and the two PID cuts together

In the simultaneous fit the signal region only enters the *starting values* (from the sideband subtraction of §3.4),
so $A$ should not depend on it. And the two PID cuts together on a two-dimensional grid.

In [ ]:
rows = []
for lo_sr in (5170.0, 5200.0, 5230.0):                   # edges of the signal region
    for hi_sr in (5320.0, 5350.0, 5380.0):
        st = dict(best_vals_start)
        for lab, m_side in (("pos", m_pos), ("neg", m_neg)):
            b_, _ = sideband_background(m_side, (lo_sr, hi_sr))        # expected background under the peak (sidebands)
            n_sr = np.sum((m_side > lo_sr) & (m_side < hi_sr))        # observed events in the signal region
            st[f"N_b_{lab}"] = len(m_side) - (n_sr - b_)              # starting background yield of the joint fit
        st["N_s"] = len(sample) - st["N_b_pos"] - st["N_b_neg"]
        f_sr = fit_joint(m_pos, m_neg, st)                            # the signal region only changed the start values
        rows.append((lo_sr, hi_sr, f_sr.values["A"]))
sr = pd.DataFrame(rows, columns=["sr_lo", "sr_hi", "A"])
print(sr.pivot(index="sr_lo", columns="sr_hi", values="A").round(5).to_string())
print(f"max |shift| over the grid of the signal region: {np.abs(sr.A - A0).max():.1e}   (report: 6e-5)")

# the two PID cuts together on a 4 x 4 grid
grid2 = pd.DataFrame([(k, p_, fit_nominal(select(k=k, pi=p_)).values["A"]) for k in (0.4, 0.5, 0.6, 0.7) for p_ in (0.3, 0.4, 0.5, 0.6)],
                     columns=["ProbK >", "ProbPi <", "A"])
print(grid2.pivot(index="ProbK >", columns="ProbPi <", values="A").round(4).to_string())
print(f"max |shift| over the PID grid: {np.abs(grid2.A - A0).max():.4f}   (report, finer grid: 0.0126)")

In [ ]:
# the three stability grids as maps of the shift with respect to the nominal fit (units of 1e-3)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
maps_ = [(1e3 * (grid_A - A0), rights, lefts, "upper edge of the window [MeV]", "lower edge of the window [MeV]"),
         (1e3 * (sr.pivot(index="sr_lo", columns="sr_hi", values="A").values - A0), [5320, 5350, 5380], [5170, 5200, 5230],
          "upper edge of the signal region [MeV]", "lower edge [MeV]"),
         (1e3 * (grid2.pivot(index="ProbK >", columns="ProbPi <", values="A").values - A0), [0.3, 0.4, 0.5, 0.6], [0.4, 0.5, 0.6, 0.7],
          "ProbPi <", "ProbK >")]
for a, (img, xt, yt, xl, yl) in zip(ax, maps_):
    im = a.imshow(img, origin="lower", aspect="auto", cmap="coolwarm", vmin=-6, vmax=6)
    a.set(xticks=range(len(xt)), xticklabels=xt, yticks=range(len(yt)), yticklabels=yt, xlabel=xl, ylabel=yl); a.grid(False)
    plt.colorbar(im, ax=a, label="shift [$10^{-3}$]")
plt.tight_layout(); save_fig("03_stability_grids")
pd.DataFrame(grid_A, index=lefts, columns=rights).to_csv("results/03_window_grid.csv")

## Results saved

In [ ]:
summary = dict(
    result=dict(A=float(A0), stat=float(s0), syst=float(syst_total)),
    groups={k: float(v) for k, v in group_unc.items()},
    sigma_B_validation=dict(spread_over_sigma_B=float(shifts.std() / np.median(sigmas))))
json.dump(summary, open("results/03_summary.json", "w"), indent=1)
syst.to_csv("results/03_systematics_variants.csv", index=False); scan.to_csv("results/03_pid_scan.csv", index=False)
sr.to_csv("results/03_signal_region_scan.csv", index=False); grid2.to_csv("results/03_pid_grid.csv", index=False)
print("saved results/03_summary.json and the tables; figures in figures/")

## Summary
* The 13 variants give a systematic uncertainty of $\approx0.007$, **dominated by the preselection** through a single
  variant (`ProbK > 0.6`); the other groups are smaller. The statistical uncertainty is $0.0085$.
* The fine scan shows that the dominant variant is a **fluctuation** (no trend on the tight side) while loosening
  `ProbK` moves $A_{\rm raw}$ monotonically, as expected if misidentified decays enter.
* The $\sigma_B$ rule is validated by bootstrap on nested samples; $A_{\rm raw}$ is stable over the grids.

In [ ]:
print(f"Running time of this notebook: {(time.time() - T0) / 60:.1f} min")